# Phase 2: GPT-2 Manual Forward Pass

*(This notebook is part of the GPT-2 Inference Engine from Scratch project.)*

## Lesson 8: HuggingFace GPT-2 Reference

**Goal:** Let's run the official HuggingFace GPT-2 model (fp32) to see the token IDs, logits shape, and the next predicted token. We will save these logits as a reference to verify our manual code later.

In [8]:
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

name = "gpt2"
tok = AutoTokenizer.from_pretrained(name)
model = AutoModelForCausalLM.from_pretrained(name, torch_dtype=torch.float32).to("cuda").eval()

prompt = "The capital of France is"
ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")
print("Token ids:", ids.tolist())
print("Tokens:", tok.convert_ids_to_tokens(ids[0]))

with torch.no_grad():
    logits = model(ids).logits

print("Logits shape:", tuple(logits.shape))
next_id = logits[0, -1].argmax().item()
print("Next token id:", next_id, "->", repr(tok.decode([next_id])))
print("Params (M):", round(sum(p.numel() for p in model.parameters()) / 1e6, 1))

torch.save({"ids": ids.cpu(), "logits": logits.cpu()}, "/kaggle/working/gpt2_reference.pt")

config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

...

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

Token ids: [[464, 3139, 286, 4881, 318]]
Tokens: ['The', 'Ġcapital', 'Ġof', 'ĠFrance', 'Ġis']
Logits shape: (1, 5, 50257)
Next token id: 262 -> ' the'
Params (M): 124.4


## Lesson 9: Inspecting GPT-2 Raw Weights

**Goal:** Before we build the model, let's inspect the raw tensors inside Transformer Block 0, check their shapes, and see the total memory footprint. This gives us a clear idea of what we are going to implement.

In [9]:
total_params = 0
total_bytes = 0

print("Global tensors:")
for name, p in model.named_parameters():
    if not name.startswith("transformer.h."):
        print(f"  {name:35s} {tuple(p.shape)}")
    total_params += p.numel()
    total_bytes += p.numel() * p.element_size()

print("\nBlock 0 tensors:")
for name, p in model.named_parameters():
    if name.startswith("transformer.h.0."):
        print(f"  {name:35s} {tuple(p.shape)}")

print(f"\nTotal params: {total_params/1e6:.1f} M")
print(f"Total fp32 size: {total_bytes/1e6:.1f} MB")
print(f"Num blocks: {model.config.n_layer}")
print(f"lm_head tied to wte:", model.lm_head.weight.data_ptr() == model.transformer.wte.weight.data_ptr())

Global tensors:
  transformer.wte.weight              (50257, 768)
  transformer.wpe.weight              (1024, 768)
  transformer.ln_f.weight             (768,)
  transformer.ln_f.bias               (768,)

Block 0 tensors:
  transformer.h.0.ln_1.weight         (768,)
  transformer.h.0.ln_1.bias           (768,)
  transformer.h.0.attn.c_attn.weight  (768, 2304)
  transformer.h.0.attn.c_attn.bias    (2304,)
  transformer.h.0.attn.c_proj.weight  (768, 768)
  transformer.h.0.attn.c_proj.bias    (768,)
  transformer.h.0.ln_2.weight         (768,)
  transformer.h.0.ln_2.bias           (768,)
  transformer.h.0.mlp.c_fc.weight     (768, 3072)
  transformer.h.0.mlp.c_fc.bias       (3072,)
  transformer.h.0.mlp.c_proj.weight   (3072, 768)
  transformer.h.0.mlp.c_proj.bias     (768,)

Total params: 124.4 M
Total fp32 size: 497.8 MB
Num blocks: 12
lm_head tied to wte: True


## Lesson 10: Raw Safetensors Inspection

**Goal:** We will read the raw `.safetensors` file directly without using any HuggingFace classes. Let's check the tensor names and verify if the `lm_head` is tied to the embeddings.

In [10]:
from huggingface_hub import hf_hub_download
from safetensors import safe_open

path = hf_hub_download("gpt2", "model.safetensors")
print("File path:", path)

with safe_open(path, framework="pt", device="cpu") as f:
    keys = sorted(f.keys())
    print("Num tensors in file:", len(keys))
    print("First 6 keys:", keys[:6])
    print("lm_head.weight in file:", "lm_head.weight" in keys)
    first_key = keys[0]
    print(f"{first_key} shape:", tuple(f.get_tensor(first_key).shape))

File path: /root/.cache/huggingface/hub/models--gpt2/snapshots/607a30d783dfa663caf39e06633721c8d4cfcd7e/model.safetensors
Num tensors in file: 160
First 6 keys: ['h.0.attn.bias', 'h.0.attn.c_attn.bias', 'h.0.attn.c_attn.weight', 'h.0.attn.c_proj.bias', 'h.0.attn.c_proj.weight', 'h.0.ln_1.bias']
lm_head.weight in file: False
h.0.attn.bias shape: (1, 1, 1024, 1024)


## Lesson 11: Manual Weight Dictionary

**Goal:** Now we will manually load all the parameter tensors into the GPU, skipping the mask buffers that we don't need. We will organize them layer by layer and verify them against the HuggingFace model.

In [11]:
import torch
from safetensors import safe_open

n_layers = model.config.n_layer
blocks = [dict() for _ in range(n_layers)]
globals_ = {}

with safe_open(path, framework="pt", device="cpu") as f:
    for key in f.keys():
        if "attn.bias" in key:  # causal mask buffer, not a weight -- skip
            continue
        tensor = f.get_tensor(key).to("cuda")
        if key.startswith("h."):
            _, layer_str, sub_name = key.split(".", 2)
            blocks[int(layer_str)][sub_name] = tensor
        else:
            globals_[key] = tensor

print("Global keys:", sorted(globals_.keys()))
print("Block 0 keys:", sorted(blocks[0].keys()))
print("Num blocks loaded:", len(blocks))

matches = torch.equal(globals_["wte.weight"], model.transformer.wte.weight.data)
print("Our wte exactly matches HF wte:", matches)

total_mb = sum(t.numel() * t.element_size() for t in globals_.values())
total_mb += sum(t.numel() * t.element_size() for b in blocks for t in b.values())
print("Total loaded size (MB):", round(total_mb / 1e6, 1))

Global keys: ['ln_f.bias', 'ln_f.weight', 'wpe.weight', 'wte.weight']
Block 0 keys: ['attn.c_attn.weight', 'attn.c_proj.bias', 'attn.c_proj.weight', 'ln_1.bias', 'ln_1.weight', 'ln_2.bias', 'ln_2.weight', 'mlp.c_fc.bias', 'mlp.c_fc.weight', 'mlp.c_proj.bias', 'mlp.c_proj.weight']
Num blocks loaded: 12
Our wte exactly matches HF wte: True
Total loaded size (MB): 497.6


## Lesson 12: Fixed Loading and Assertions

**Goal:** Just to be absolutely sure, we are adding an exact-match filter to correctly skip the causal mask buffers. We will also add assertions for the block count and tensor values to make sure our loading logic is bug-free.

In [12]:
import torch
from safetensors import safe_open

n_layers = model.config.n_layer
blocks = [dict() for _ in range(n_layers)]
globals_ = {}

with safe_open(path, framework="pt", device="cpu") as f:
    for key in f.keys():
        if key.startswith("h."):
            _, layer_str, sub_name = key.split(".", 2)
        else:
            sub_name = key
        if sub_name == "attn.bias":  # exact match only -- causal mask buffer
            continue
        tensor = f.get_tensor(key).to("cuda")
        if key.startswith("h."):
            blocks[int(layer_str)][sub_name] = tensor
        else:
            globals_[key] = tensor

for i, b in enumerate(blocks):
    assert len(b) == 12, f"block {i} has {len(b)} tensors, expected 12"
total_count = len(globals_) + sum(len(b) for b in blocks)
assert total_count == 148, f"total {total_count}, expected 148"
print("All 12 blocks have 12 tensors. Total:", total_count)

checks = {
    "wte.weight": (globals_["wte.weight"], model.transformer.wte.weight.data),
    "block0 attn.c_attn.bias": (blocks[0]["attn.c_attn.bias"], model.transformer.h[0].attn.c_attn.bias.data),
    "block11 mlp.c_proj.weight": (blocks[11]["mlp.c_proj.weight"], model.transformer.h[11].mlp.c_proj.weight.data),
}
for name, (ours, theirs) in checks.items():
    print(f"{name}: exact match = {torch.equal(ours, theirs)}")

All 12 blocks have 12 tensors. Total: 148
wte.weight: exact match = True
block0 attn.c_attn.bias: exact match = True
block11 mlp.c_proj.weight: exact match = True


## Lesson 13: Embedding Lookup

**Goal:** Let's manually compute the Token Embeddings and Position Embeddings, add them together, and compare our result directly with HuggingFace's embedding modules.

In [13]:
ids = tok(prompt, return_tensors="pt").input_ids.to("cuda")[0]
seq_len = ids.shape[0]
positions = torch.arange(seq_len, device="cuda")

x_manual = globals_["wte.weight"][ids] + globals_["wpe.weight"][positions]

with torch.no_grad():
    x_ref = model.transformer.wte(ids) + model.transformer.wpe(positions)

print("Manual embedding shape:", tuple(x_manual.shape))
print("Exact match with HF:", torch.equal(x_manual, x_ref))
print("Row 0, first 5 values:", x_manual[0, :5].tolist())

Manual embedding shape: (5, 768)
Exact match with HF: True
Row 0, first 5 values: [-0.08744361251592636, -0.21771633625030518, 0.06849727779626846, -0.05073772370815277, -0.049695029854774475]


## Lesson 14: Manual LayerNorm

**Goal:** We are going to write the math for LayerNorm (mean, variance, scale, and shift) completely from scratch for Block 0, and then verify it against the native PyTorch LayerNorm.

In [14]:
def manual_layernorm(x, weight, bias, eps):
    mean = x.mean(dim=-1, keepdim=True)
    var = x.var(dim=-1, unbiased=False, keepdim=True)
    return (x - mean) / torch.sqrt(var + eps) * weight + bias

eps = model.config.layer_norm_epsilon
ln1_manual = manual_layernorm(x_manual, blocks[0]["ln_1.weight"], blocks[0]["ln_1.bias"], eps)

with torch.no_grad():
    ln1_ref = model.transformer.h[0].ln_1(x_ref)

print("eps:", eps)
print("Exact match:", torch.equal(ln1_manual, ln1_ref))
print("Max abs diff:", (ln1_manual - ln1_ref).abs().max().item())
print("Row 0, first 5 values (manual):", ln1_manual[0, :5].tolist())

eps: 1e-05
Exact match: False
Max abs diff: 8.940696716308594e-08
Row 0, first 5 values (manual): [-0.052923526614904404, -0.07779036462306976, -0.03243910148739815, -0.028004392981529236, -0.03955240547657013]


## Lesson 15: QKV Projection and Head Split

**Goal:** Here we extract the Query, Key, and Value matrices from the `c_attn` projection and split them into 12 attention heads. We will compare our manual splitting logic with the reference.

In [15]:
def split_heads(x, n_head):
    seq_len, dim = x.shape
    head_dim = dim // n_head
    return x.view(seq_len, n_head, head_dim).transpose(0, 1)  # (n_head, seq_len, head_dim)

n_head = model.config.n_head
qkv_manual = ln1_manual @ blocks[0]["attn.c_attn.weight"] + blocks[0]["attn.c_attn.bias"]
q_manual, k_manual, v_manual = qkv_manual.split(768, dim=-1)
q_manual, k_manual, v_manual = (split_heads(t, n_head) for t in (q_manual, k_manual, v_manual))

with torch.no_grad():
    qkv_ref = model.transformer.h[0].attn.c_attn(ln1_manual)
q_ref, k_ref, v_ref = qkv_ref.split(768, dim=-1)
q_ref, k_ref, v_ref = (split_heads(t, n_head) for t in (q_ref, k_ref, v_ref))

print("n_head:", n_head, "head_dim:", 768 // n_head)
print("Q shape:", tuple(q_manual.shape))
print("Q max abs diff:", (q_manual - q_ref).abs().max().item())
print("K max abs diff:", (k_manual - k_ref).abs().max().item())
print("V max abs diff:", (v_manual - v_ref).abs().max().item())

n_head: 12 head_dim: 64
Q shape: (12, 5, 64)
Q max abs diff: 0.0
K max abs diff: 0.0
V max abs diff: 0.0


## Lesson 16: Manual Causal Self-Attention

**Goal:** Using our Q, K, and V tensors, we will write the full Causal Self-Attention math manually (scaling, masking, softmax, and multiplying with V) and verify it against PyTorch's `scaled_dot_product_attention`.

In [16]:
import math

def manual_attention(q, k, v):
    head_dim = q.shape[-1]
    seq_len = q.shape[-2]
    scores = q @ k.transpose(-2, -1) / math.sqrt(head_dim)
    mask = torch.triu(torch.ones(seq_len, seq_len, device=q.device, dtype=torch.bool), diagonal=1)
    scores = scores.masked_fill(mask, float("-inf"))
    probs = torch.softmax(scores, dim=-1)
    return probs @ v, probs

attn_out_manual, probs = manual_attention(q_manual, k_manual, v_manual)

with torch.no_grad():
    attn_out_ref = torch.nn.functional.scaled_dot_product_attention(q_manual, k_manual, v_manual, is_causal=True)

print("Output shape:", tuple(attn_out_manual.shape))
print("Max abs diff vs sdpa:", (attn_out_manual - attn_out_ref).abs().max().item())
print("Head 0, token 0 attention weights:", probs[0, 0].tolist())
print("Head 0, token 4 attention weights:", probs[0, 4].tolist())

Output shape: (12, 5, 64)
Max abs diff vs sdpa: 1.1920928955078125e-07
Head 0, token 0 attention weights: [1.0, 0.0, 0.0, 0.0, 0.0]
Head 0, token 4 attention weights: [0.6099013090133667, 0.15188203752040863, 0.06255988776683807, 0.09449265152215958, 0.08116415143013]


## Lesson 17: Merge Heads, Output Projection, and Residual

**Goal:** Time to merge the 12 attention heads back together, apply the final `c_proj` linear layer, and add the residual connection. We will verify if our split-and-merge logic is mathematically correct.

In [17]:
def merge_heads(x):
    n_head, seq_len, head_dim = x.shape
    return x.transpose(0, 1).contiguous().view(seq_len, n_head * head_dim)

q_roundtrip = split_heads(merge_heads(q_manual), n_head)
print("Split/merge round-trip exact:", torch.equal(q_roundtrip, q_manual))

attn_merged = merge_heads(attn_out_manual)
attn_proj = attn_merged @ blocks[0]["attn.c_proj.weight"] + blocks[0]["attn.c_proj.bias"]
x_after_attn = x_manual + attn_proj

print("Merged shape:", tuple(attn_merged.shape))
print("x_after_attn shape:", tuple(x_after_attn.shape))
print("x_manual mean abs value:", x_manual.abs().mean().item())
print("attn_proj mean abs value:", attn_proj.abs().mean().item())

Split/merge round-trip exact: True
Merged shape: (5, 768)
x_after_attn shape: (5, 768)
x_manual mean abs value: 0.11914384365081787
attn_proj mean abs value: 0.29924681782722473


## Lesson 18: The MLP Block

**Goal:** We are implementing the MLP block now. This includes the second LayerNorm, expanding via `c_fc`, applying the GELU activation, projecting back via `c_proj`, and adding the residual connection.

In [18]:
act_name = model.config.activation_function
approx = "tanh" if "new" in act_name else "none"
print("Activation function:", act_name, "-> approximate=", approx)

ln2_manual = manual_layernorm(x_after_attn, blocks[0]["ln_2.weight"], blocks[0]["ln_2.bias"], eps)
mlp_hidden = ln2_manual @ blocks[0]["mlp.c_fc.weight"] + blocks[0]["mlp.c_fc.bias"]
mlp_hidden = torch.nn.functional.gelu(mlp_hidden, approximate=approx)
mlp_out = mlp_hidden @ blocks[0]["mlp.c_proj.weight"] + blocks[0]["mlp.c_proj.bias"]
x_block0_manual = x_after_attn + mlp_out

with torch.no_grad():
    mlp_out_ref = model.transformer.h[0].mlp(model.transformer.h[0].ln_2(x_after_attn))
    x_block0_ref = x_after_attn + mlp_out_ref

print("mlp_hidden shape:", tuple(mlp_hidden.shape))
print("Max abs diff (mlp_out):", (mlp_out - mlp_out_ref).abs().max().item())
print("Max abs diff (full block 0 output):", (x_block0_manual - x_block0_ref).abs().max().item())

Activation function: gelu_new -> approximate= tanh
mlp_hidden shape: (5, 3072)
Max abs diff (mlp_out): 1.1444091796875e-05
Max abs diff (full block 0 output): 1.1444091796875e-05


## Lesson 19: Full Model Forward Pass

**Goal:** Let's put it all together! We will stack all 12 manual blocks in a loop, apply the final LayerNorm, and multiply by the `lm_head`. Finally, we compare our logits with the HuggingFace reference from Lesson 8.

In [19]:
def transformer_block(x, block, n_head, eps):
    ln1 = manual_layernorm(x, block["ln_1.weight"], block["ln_1.bias"], eps)
    qkv = ln1 @ block["attn.c_attn.weight"] + block["attn.c_attn.bias"]
    q, k, v = (split_heads(t, n_head) for t in qkv.split(768, dim=-1))
    attn_out, _ = manual_attention(q, k, v)
    attn_out = merge_heads(attn_out) @ block["attn.c_proj.weight"] + block["attn.c_proj.bias"]
    x = x + attn_out

    ln2 = manual_layernorm(x, block["ln_2.weight"], block["ln_2.bias"], eps)
    h = ln2 @ block["mlp.c_fc.weight"] + block["mlp.c_fc.bias"]
    h = torch.nn.functional.gelu(h, approximate="tanh")
    mlp_out = h @ block["mlp.c_proj.weight"] + block["mlp.c_proj.bias"]
    return x + mlp_out

x = x_manual
for block in blocks:
    x = transformer_block(x, block, n_head, eps)

x = manual_layernorm(x, globals_["ln_f.weight"], globals_["ln_f.bias"], eps)
logits_manual = x @ globals_["wte.weight"].T

ref = logits[0]  # (5, 50257), still in GPU memory from Lesson 8
print("logits_manual shape:", tuple(logits_manual.shape))
print("Max abs diff vs reference:", (logits_manual - ref).abs().max().item())

next_id = logits_manual[-1].argmax().item()
print("Predicted next token (manual):", next_id, "->", repr(tok.decode([next_id])))
print("Match with HF (' the', id 262):", next_id == 262)

logits_manual shape: (5, 50257)
Max abs diff vs reference: 6.103515625e-05
Predicted next token (manual): 262 -> ' the'
Match with HF (' the', id 262): True


## Lesson 20: Naive Generation (No KV Cache)

**Goal:** We are building a basic generation loop without any KV caching. It generates 10 tokens by running the entire forward pass from scratch at every single step, showing us exactly why naive generation is so slow.

In [20]:
import time

def forward_full(ids):
    seq_len = ids.shape[0]
    positions = torch.arange(seq_len, device="cuda")
    x = globals_["wte.weight"][ids] + globals_["wpe.weight"][positions]
    for block in blocks:
        x = transformer_block(x, block, n_head, eps)
    x = manual_layernorm(x, globals_["ln_f.weight"], globals_["ln_f.bias"], eps)
    return x @ globals_["wte.weight"].T

gen_ids = ids.clone()
times = []
for step in range(10):
    torch.cuda.synchronize()
    start = time.perf_counter()
    logits_step = forward_full(gen_ids)
    torch.cuda.synchronize()
    times.append(time.perf_counter() - start)
    next_id = logits_step[-1].argmax().item()
    gen_ids = torch.cat([gen_ids, torch.tensor([next_id], device="cuda")])

print("Generated:", repr(tok.decode(gen_ids.tolist())))
print("Seq lengths:", [5 + i for i in range(10)])
print("Time per step (ms):", [round(t * 1000, 3) for t in times])

Generated: 'The capital of France is the capital of the French Republic, and the capital'
Seq lengths: [5, 6, 7, 8, 9, 10, 11, 12, 13, 14]
Time per step (ms): [10.328, 11.179, 10.633, 10.301, 10.176, 9.938, 11.008, 10.296, 10.911, 10.331]
